# Omni-Fusion: Phase 1 - Data Acquisition

This notebook sets up the environment and acquires the PTB-XL and MIMIC-IV Clinical Demo datasets from PhysioNet.

Requirements:
- Download PTB-XL (v1.0.3)
- Download MIMIC-IV Clinical Demo (v2.2)
- Verify file counts and checksums
- Load and display a sample record from each dataset


In [1]:
# 1. Setup Environment
!pip install -q wfdb "pandas<2.4.0"

import os
import subprocess
import pandas as pd
import numpy as np
import wfdb

def run_cmd(cmd):
    print(f"Running: {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error running '{cmd}': {result.stderr}")
    return result

def verify_checksums(directory):
    checksum_files = []
    for root, dirs, files in os.walk(directory):
        if 'SHA256SUMS.txt' in files:
            checksum_files.append(os.path.join(root, 'SHA256SUMS.txt'))
    
    if not checksum_files:
        print(f"No SHA256SUMS.txt found in {directory}")
        return False
    
    checksum_file = checksum_files[0]
    base_dir = os.path.dirname(checksum_file)
    print(f"Verifying checksums using {checksum_file}...")
    
    cmd = f"cd '{base_dir}' && shasum -a 256 -c SHA256SUMS.txt"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    success_count = result.stdout.count("OK")
    failure_count = result.stdout.count("FAILED")
    print(f"Checksum verification: {success_count} OK, {failure_count} FAILED")
    
    if failure_count > 0:
        return False
    return True

def count_files(directory):
    count = sum(len(files) for _, _, files in os.walk(directory))
    print(f"Total files in {directory}: {count}")
    return count


In [2]:
# 2. Acquire and Verify PTB-XL
print("Setting up PTB-XL...")
dest_dir = "data/raw/ptbxl"
os.makedirs(dest_dir, exist_ok=True)
zip_path = os.path.join(dest_dir, "ptbxl.zip")

if not os.path.exists(zip_path):
    url = "https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip"
    run_cmd(f"curl -L -s -o '{zip_path}' '{url}'")

if not os.path.exists(os.path.join(dest_dir, "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")):
    print("Unzipping PTB-XL...")
    run_cmd(f"unzip -q '{zip_path}' -d '{dest_dir}'")

extracted_dir = os.path.join(dest_dir, "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")
count_files(extracted_dir)
verify_checksums(extracted_dir)

# Load sample
csv_path = os.path.join(extracted_dir, "ptbxl_database.csv")
df = pd.read_csv(csv_path)
print(f"PTB-XL Metadata shape: {df.shape}")
print(f"PTB-XL Metadata columns: {list(df.columns)}")

sample_record = df.iloc[0]
record_path = os.path.join(extracted_dir, sample_record.filename_hr)
record = wfdb.rdrecord(record_path)
print(f"PTB-XL Sample Signal shape: {record.p_signal.shape}")


Setting up PTB-XL...
Running: curl -L -s -o 'data/raw/ptbxl/ptbxl.zip' 'https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip'
Unzipping PTB-XL...
Running: unzip -q 'data/raw/ptbxl/ptbxl.zip' -d 'data/raw/ptbxl'
Error running 'unzip -q 'data/raw/ptbxl/ptbxl.zip' -d 'data/raw/ptbxl'': shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
unzip:  cannot find or open data/raw/ptbxl/ptbxl.zip, data/raw/ptbxl/ptbxl.zip.zip or data/raw/ptbxl/ptbxl.zip.ZIP.

Total files in data/raw/ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3: 0
No SHA256SUMS.txt found in data/raw/ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3


FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/ptbxl_database.csv'

In [ ]:
# 3. Acquire and Verify MIMIC-IV Clinical Demo
print("Setting up MIMIC-IV Clinical Demo...")
dest_dir = "data/raw/mimic_iv_demo"
os.makedirs(dest_dir, exist_ok=True)
zip_path = os.path.join(dest_dir, "mimic.zip")

if not os.path.exists(zip_path):
    url = "https://physionet.org/static/published-projects/mimic-iv-demo/mimic-iv-clinical-database-demo-2.2.zip"
    run_cmd(f"curl -L -s -o '{zip_path}' '{url}'")

if not os.path.exists(os.path.join(dest_dir, "mimic-iv-clinical-database-demo-2.2")):
    print("Unzipping MIMIC-IV Demo...")
    run_cmd(f"unzip -q '{zip_path}' -d '{dest_dir}'")
    
extracted_dir = os.path.join(dest_dir, "mimic-iv-clinical-database-demo-2.2")
count_files(extracted_dir)
verify_checksums(extracted_dir)

# Load sample
csv_path = os.path.join(extracted_dir, "hosp", "patients.csv")
df = pd.read_csv(csv_path)
print(f"MIMIC-IV Patients shape: {df.shape}")
print(f"MIMIC-IV Patients columns: {list(df.columns)}")
